# Apache Iceberg on Snowflake: complete ecommerce lifecycle

This notebook begins with Iceberg's origin and design, then builds an ecommerce lakehouse with `ICEBERG_PRODUCTS`, `ICEBERG_INVENTORY`, `ICEBERG_ORDERS`, `ICEBERG_ORDER_ITEMS`, and `ICEBERG_PAYMENTS`. The labs cover creation, reads, inserts, updates, deletes, ACID transactions, rollback, deterministic `MERGE`, Time Travel, recovery, cloning, schema evolution, partitioning, v3, streams, compaction, retention, external catalogs, and drop recovery.

Every fenced block targets a Snowflake SQL worksheet. Run normal blocks in order. Run blocks marked **intentional error** separately. The runnable path uses Snowflake-provided storage; customer-storage and external-catalog examples are commented templates because their IAM configuration is account-specific.

## 1. Where Apache Iceberg came from

Apache Iceberg began at Netflix in 2017. Ryan Blue and Daniel Weeks are commonly credited as its creators, working with the Netflix data platform team to replace fragile Hive-style table layouts at very large scale. Netflix open-sourced the work and proposed it to the Apache Software Foundation in 2018; the proposal lists Blue and Weeks among the Netflix initial committers. Iceberg then developed under vendor-neutral Apache governance.

The original problem was larger than storing Parquet files. Data engines needed a dependable answer to: **Which files, schema, and partition specification form the table at this instant?** Object-store directory listing was slow and could not safely coordinate concurrent changes. Hive partition columns were exposed to users, renames could be confused with drop-and-add, and multi-file updates could leave partial states.

Iceberg made table metadata a first-class, versioned layer. This enabled several engines to work with one logical table while preserving consistent snapshots. Sources: [Apache Iceberg incubation proposal](https://cwiki.apache.org/confluence/spaces/INCUBATOR/pages/109454835/IcebergProposal), [ASF history overview](https://news.apache.org/foundation/entry/asf-project-spotlight-apache-iceberg), [original Netflix repository](https://github.com/Netflix/iceberg).

## 2. Iceberg is a table format, not a file format or query engine

| Layer | Ecommerce example | Responsibility |
|---|---|---|
| Query engine | Snowflake, Spark, Flink, Trino | Plans and executes SQL/processing |
| Catalog | Snowflake catalog, REST catalog, Glue, Open Catalog | Atomically identifies the current table metadata |
| Table format | Apache Iceberg | Defines schemas, partition specs, snapshots, manifests, and commit rules |
| File format | Apache Parquet | Encodes rows in columnar data files |
| Object storage | Snowflake storage, S3, Azure, GCS | Stores data and metadata files |

Parquet alone does not provide table transactions or schema history. A catalog alone does not define Iceberg's snapshot graph. Iceberg coordinates these layers without becoming a compute engine. Snowflake currently reads and writes Iceberg table data as Parquet. Source: [Snowflake Iceberg overview](https://docs.snowflake.com/en/user-guide/tables-iceberg).

## 3. Follow one Iceberg snapshot

```text
Catalog pointer
    |
    v
metadata.json
  - current schema and field IDs
  - partition specifications
  - table properties
  - snapshot log and current snapshot ID
    |
    v
manifest list for snapshot
    |
    v
manifests
  - data/delete file paths
  - partitions and column statistics
  - record counts and file state
    |
    +--> immutable Parquet data files
    +--> positional delete files or deletion vectors when applicable
```

A writer creates new data/delete/metadata files and then atomically commits new table metadata through the catalog. Existing readers continue using their chosen snapshot. Failed or conflicting commits do not expose half of a new file set. Old snapshots retain references needed for historical reads until expiry.

Metadata pruning happens in stages: snapshot to manifest list, manifests to candidate files, then Parquet statistics and engine filters. This avoids listing every object in a large table.

## 4. Major Iceberg features and why they matter

| Feature | What it solves | Ecommerce example |
|---|---|---|
| Atomic snapshots | Multi-file changes publish as one table state | Order batch appears completely |
| Consistent concurrent reads | Readers stay on a valid snapshot while writers commit | BI query does not see half a checkout batch |
| Schema evolution with field IDs | Rename/add/drop without identifying columns only by position/name | Rename `STATUS` without treating it as a new field |
| Hidden partitioning | Users filter business columns; engines derive partition values | Filter `ORDER_TS` while files are partitioned by day |
| Partition evolution | New layouts can coexist with old specs in core Iceberg | Move from day to month without rewriting all old data |
| Time travel | Query retained historical snapshots | Inspect catalog before a bad price update |
| Row-level changes | Updates/deletes through copy-on-write or merge-on-read | Cancel or return an order |
| Metadata statistics | Plan without object-store directory listing | Prune dates and value ranges |
| Multi-engine interoperability | Several compatible engines use one table contract | Snowflake analytics plus Spark processing |
| Format versions | Add capabilities while preserving explicit compatibility | V2 positional deletes; v3 deletion vectors/row lineage |

Snowflake support depends on catalog mode, storage mode, format version, region, edition, and feature status. Core Iceberg capability does not automatically mean every Snowflake-managed or externally managed configuration exposes the same operation.

## 5. Choose catalog and storage ownership first

| Snowflake configuration | Catalog owner | File storage | Writes and maintenance |
|---|---|---|---|
| Snowflake catalog + Snowflake storage | Snowflake | Snowflake-provided | Full DML; Snowflake manages files and lifecycle |
| Snowflake catalog + customer external volume | Snowflake | Your bucket/container | Full DML; Snowflake manages Iceberg lifecycle, you protect storage |
| Writable remote REST catalog | External catalog | Customer storage | Writes depend on integration; coordinate every engine through the catalog |
| Glue/files/Delta registration | External system/files | Customer storage | Commonly read-oriented; external owner maintains and Snowflake refreshes |
| Catalog-linked database | External catalog namespace | Customer storage | Capability depends on catalog and write configuration |

`EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'` is a reserved storage choice, not an object created with `CREATE EXTERNAL VOLUME`. It is currently limited by cloud/region availability. If unavailable, use the customer external-volume template in section 25. Do not combine clauses from different `CREATE ICEBERG TABLE` variants. Source: [Iceberg storage choices](https://docs.snowflake.com/en/user-guide/tables-iceberg-storage).

## 6. Select the database and create the Iceberg schema

Select an existing writable database and a role with schema/table creation privileges in Snowsight. Use the existing `SNOWFLAKE_LEARNING_WH`; table creation requires usable warehouse compute.

```sql
CREATE SCHEMA IcebergTableLifecycle;
USE SCHEMA IcebergTableLifecycle;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;

SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE(), CURRENT_ROLE();
```

All lab tables, streams and recovery objects are created in `IcebergTableLifecycle`. Replace that schema name consistently with your team's choice if needed. Keep the same database selected throughout. Setup runs once in a new schema; later sections depend on earlier data and variables.

The main path uses Snowflake storage via `EXTERNAL_VOLUME='SNOWFLAKE_MANAGED'`, so it needs no personal S3 bucket or AWS access keys. Check support in your account before starting; customer-managed storage remains an optional administrator-configured alternative. [Snowflake storage for Iceberg](https://docs.snowflake.com/en/user-guide/tables-iceberg-internal-storage)

## 7. Create Snowflake-managed Iceberg ecommerce tables

The main tables use Iceberg v2, the compatibility-oriented default. `ICEBERG_ORDERS` uses hidden day partitioning. `CHANGE_TRACKING` supports the later stream example. `AUTO` lets Snowflake choose row-level write behavior and target file size.

```sql
CREATE ICEBERG TABLE ICEBERG_PRODUCTS (
  SKU STRING NOT NULL, PRODUCT_NAME STRING NOT NULL, CATEGORY STRING NOT NULL,
  UNIT_PRICE DECIMAL(12,2) NOT NULL, IS_ACTIVE BOOLEAN NOT NULL, UPDATED_AT TIMESTAMP_LTZ(6) NOT NULL
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED'
  DATA_RETENTION_TIME_IN_DAYS=1 TARGET_FILE_SIZE='AUTO' ENABLE_DATA_COMPACTION=TRUE;

CREATE ICEBERG TABLE ICEBERG_INVENTORY (
  SKU STRING NOT NULL, ON_HAND LONG NOT NULL, RESERVED LONG NOT NULL, UPDATED_AT TIMESTAMP_LTZ(6) NOT NULL
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED' DATA_RETENTION_TIME_IN_DAYS=1;

CREATE ICEBERG TABLE ICEBERG_ORDERS (
  ORDER_ID LONG NOT NULL, CUSTOMER_ID LONG NOT NULL, ORDER_TS TIMESTAMP_NTZ(6) NOT NULL,
  STATUS STRING NOT NULL, ORDER_TOTAL DECIMAL(12,2) NOT NULL, SOURCE_VERSION LONG NOT NULL,
  UPDATED_AT TIMESTAMP_LTZ(6) NOT NULL
) PARTITION BY (DAY(ORDER_TS))
  CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED'
  DATA_RETENTION_TIME_IN_DAYS=1 CHANGE_TRACKING=TRUE
  TARGET_FILE_SIZE='AUTO' ENABLE_DATA_COMPACTION=TRUE
  ICEBERG_MERGE_ON_READ_BEHAVIOR='AUTO';

CREATE ICEBERG TABLE ICEBERG_ORDER_ITEMS (
  ORDER_ID LONG NOT NULL, SKU STRING NOT NULL, QUANTITY LONG NOT NULL,
  UNIT_PRICE DECIMAL(12,2) NOT NULL, LINE_TOTAL DECIMAL(12,2) NOT NULL
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED' DATA_RETENTION_TIME_IN_DAYS=1;

CREATE ICEBERG TABLE ICEBERG_PAYMENTS (
  PAYMENT_ID LONG NOT NULL, ORDER_ID LONG NOT NULL, PAYMENT_STATUS STRING NOT NULL,
  AMOUNT DECIMAL(12,2) NOT NULL, PAID_AT TIMESTAMP_LTZ(6)
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED' DATA_RETENTION_TIME_IN_DAYS=1;
```

If partitioned writes are unavailable in the account, remove only the `PARTITION BY` clause and recreate `ICEBERG_ORDERS`. Iceberg constraints and enforcement vary by table/catalog mode; the lab relies on `NOT NULL` and explicit data-quality logic rather than assuming primary-key enforcement.

## 8. Seed data and inspect the table definitions

```sql
INSERT INTO ICEBERG_PRODUCTS VALUES
  ('ICE-SKU-LAP-01', 'Developer Laptop', 'COMPUTERS', 1200.00, TRUE, CURRENT_TIMESTAMP()),
  ('ICE-SKU-MOU-01', 'Wireless Mouse', 'ACCESSORIES', 35.00, TRUE, CURRENT_TIMESTAMP()),
  ('ICE-SKU-HDP-01', 'USB-C Headphones', 'AUDIO', 80.00, TRUE, CURRENT_TIMESTAMP());

INSERT INTO ICEBERG_INVENTORY VALUES
  ('ICE-SKU-LAP-01', 10, 0, CURRENT_TIMESTAMP()),
  ('ICE-SKU-MOU-01', 50, 0, CURRENT_TIMESTAMP()),
  ('ICE-SKU-HDP-01', 25, 0, CURRENT_TIMESTAMP());

SELECT P.SKU, P.PRODUCT_NAME, P.UNIT_PRICE, I.ON_HAND, I.RESERVED
FROM ICEBERG_PRODUCTS P JOIN ICEBERG_INVENTORY I USING (SKU) ORDER BY SKU;

SHOW ICEBERG TABLES IN SCHEMA IcebergTableLifecycle;
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
DESCRIBE ICEBERG TABLE ICEBERG_ORDERS;
SELECT GET_DDL('ICEBERG_TABLE', 'IcebergTableLifecycle.ICEBERG_ORDERS');
```

`SHOW ICEBERG TABLES` exposes catalog, external volume, table type, refresh status, format version, and partition specifications when supported. `GET_DDL` is useful for source control, but secrets and account-specific identifiers should be parameterized before reuse.

## 9. Generate current Iceberg metadata

Snowflake commits DML to its catalog immediately. It also generates Iceberg metadata periodically so external engines can resolve the table. This system function requests current metadata generation on demand.

```sql
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION(
  'IcebergTableLifecycle.ICEBERG_ORDERS'
) AS ICEBERG_METADATA_RESULT;
```

An external engine reads Iceberg metadata and catalog state rather than Snowflake Query History. Publishing metadata is therefore a separate concern from committing SQL in Snowflake. With Snowflake Open Catalog or Horizon catalog access, external engines can resolve compatible Snowflake-managed tables according to current feature support.

## 10. Insert, update, delete, and truncate semantics

Logical SQL looks familiar, but physical row changes create new data/delete files and a new metadata state. The catalog pointer changes only after commit.

```sql
INSERT INTO ICEBERG_PRODUCTS VALUES
  ('ICE-SKU-KEY-01', 'Mechanical Keyboard', 'ACCESSORIES', 95.00, TRUE, CURRENT_TIMESTAMP());
INSERT INTO ICEBERG_INVENTORY VALUES
  ('ICE-SKU-KEY-01', 15, 0, CURRENT_TIMESTAMP());

UPDATE ICEBERG_PRODUCTS
SET UNIT_PRICE = 90.00, UPDATED_AT = CURRENT_TIMESTAMP()
WHERE SKU = 'ICE-SKU-KEY-01';

DELETE FROM ICEBERG_INVENTORY WHERE SKU = 'ICE-SKU-KEY-01';
DELETE FROM ICEBERG_PRODUCTS WHERE SKU = 'ICE-SKU-KEY-01';

SELECT * FROM ICEBERG_PRODUCTS ORDER BY SKU;
```

Snowflake-managed Iceberg tables support full DML including `INSERT`, `UPDATE`, `DELETE`, `MERGE`, and `TRUNCATE`. External-catalog write support depends on the catalog integration and its write configuration. Source: [manage Iceberg tables](https://docs.snowflake.com/en/user-guide/tables-iceberg-manage).

## 11. Statement atomicity

The null `SKU` violates an enforced `NOT NULL` constraint. Run the error block separately: neither tuple is committed, so a partial new data-file set never becomes the current snapshot.

```sql
SELECT COUNT(*) AS ROWS_BEFORE FROM ICEBERG_PRODUCTS;
```

**Intentional error - run separately:**

```sql
INSERT INTO ICEBERG_PRODUCTS VALUES
  ('ICE-SKU-WEB-01', 'Web Camera', 'ACCESSORIES', 70.00, TRUE, CURRENT_TIMESTAMP()),
  (NULL, 'Invalid Product', 'TEST', 1.00, TRUE, CURRENT_TIMESTAMP());
```

```sql
SELECT COUNT(*) AS ROWS_AFTER FROM ICEBERG_PRODUCTS;
SELECT * FROM ICEBERG_PRODUCTS WHERE SKU = 'ICE-SKU-WEB-01';
```

## 12. Multi-table ACID checkout

Snowflake groups these DML statements into one atomic transaction even though several Iceberg tables and many physical files can be involved. Readers see the old committed state or the complete new state.

```sql
BEGIN TRANSACTION;
  UPDATE ICEBERG_INVENTORY
     SET ON_HAND = ON_HAND - 1, UPDATED_AT = CURRENT_TIMESTAMP()
   WHERE SKU = 'ICE-SKU-LAP-01' AND ON_HAND - RESERVED >= 1;

  INSERT INTO ICEBERG_ORDERS VALUES
    (6001, 201, CURRENT_TIMESTAMP(), 'PAID', 1200.00, 1, CURRENT_TIMESTAMP());
  INSERT INTO ICEBERG_ORDER_ITEMS VALUES
    (6001, 'ICE-SKU-LAP-01', 1, 1200.00, 1200.00);
  INSERT INTO ICEBERG_PAYMENTS VALUES
    (9601, 6001, 'CAPTURED', 1200.00, CURRENT_TIMESTAMP());

  SELECT * FROM ICEBERG_ORDERS WHERE ORDER_ID = 6001;
COMMIT;

SELECT O.ORDER_ID, O.STATUS, O.ORDER_TOTAL, I.SKU, I.QUANTITY, P.PAYMENT_STATUS
FROM ICEBERG_ORDERS O
JOIN ICEBERG_ORDER_ITEMS I ON I.ORDER_ID = O.ORDER_ID
JOIN ICEBERG_PAYMENTS P ON P.ORDER_ID = O.ORDER_ID
WHERE O.ORDER_ID = 6001;
```

Production checkout code must verify the inventory update affected exactly one row before inserting the order. Use application logic or Snowflake Scripting `SQLROWCOUNT`, and roll back on zero. Format-level ACID does not replace business invariant checks.

## 13. Transaction rollback

The intermediate reads see the transaction's own changes. `ROLLBACK` discards all of them and leaves the previous snapshots current.

```sql
BEGIN TRANSACTION;
  UPDATE ICEBERG_INVENTORY SET ON_HAND = ON_HAND - 2, UPDATED_AT = CURRENT_TIMESTAMP()
  WHERE SKU = 'ICE-SKU-MOU-01' AND ON_HAND - RESERVED >= 2;
  INSERT INTO ICEBERG_ORDERS VALUES
    (6002, 202, CURRENT_TIMESTAMP(), 'PAYMENT_PENDING', 70.00, 1, CURRENT_TIMESTAMP());
  INSERT INTO ICEBERG_ORDER_ITEMS VALUES
    (6002, 'ICE-SKU-MOU-01', 2, 35.00, 70.00);
  INSERT INTO ICEBERG_PAYMENTS VALUES
    (9602, 6002, 'DECLINED', 70.00, CURRENT_TIMESTAMP());

  SELECT * FROM ICEBERG_ORDERS WHERE ORDER_ID = 6002;
ROLLBACK;

SELECT * FROM ICEBERG_ORDERS WHERE ORDER_ID = 6002;
SELECT * FROM ICEBERG_PAYMENTS WHERE PAYMENT_ID = 9602;
SELECT * FROM ICEBERG_INVENTORY WHERE SKU = 'ICE-SKU-MOU-01';
```

## 14. Snowflake transaction behavior versus generic Iceberg

Snowflake SQL currently uses `READ COMMITTED` visibility. Each statement sees committed data as of its start plus earlier writes from its transaction. Generic Iceberg engines often describe serializable or snapshot isolation; do not assume their session semantics are identical to Snowflake SQL.

Snowflake DDL commits separately. An `ALTER ICEBERG TABLE` encountered inside an explicit transaction first commits preceding DML, commits the DDL independently, subsequent DML follows the session transaction settings unless another explicit transaction is started. `ALTER ICEBERG TABLE ... REFRESH` is also its own committed transaction. Keep DDL/refresh outside business DML blocks.

```sql
BEGIN;
  INSERT INTO ICEBERG_ORDERS VALUES
    (6099, 299, CURRENT_TIMESTAMP(), 'DDL_TRAP', 1.00, 1, CURRENT_TIMESTAMP());
  ALTER ICEBERG TABLE ICEBERG_PRODUCTS ADD COLUMN DDL_TRAP_NOTE STRING;
ROLLBACK;

SELECT * FROM ICEBERG_ORDERS WHERE ORDER_ID = 6099; -- committed by DDL boundary
DELETE FROM ICEBERG_ORDERS WHERE ORDER_ID = 6099;
ALTER ICEBERG TABLE ICEBERG_PRODUCTS DROP COLUMN DDL_TRAP_NOTE;
```

Source: [Iceberg tables and Snowflake transactions](https://docs.snowflake.com/en/user-guide/tables-iceberg-transactions).

## 15. Deterministic, idempotent `MERGE`

The replay table is itself a transient Iceberg table. Deduplicate one latest event per order and accept only newer source versions. Delete tombstones do not create previously absent rows.

```sql
CREATE OR REPLACE TRANSIENT ICEBERG TABLE ICEBERG_ORDER_EVENT_BATCH (
  ORDER_ID LONG, CUSTOMER_ID LONG, ORDER_TS TIMESTAMP_NTZ(6), STATUS STRING,
  ORDER_TOTAL DECIMAL(12,2), SOURCE_VERSION LONG, OP STRING, EVENT_TS TIMESTAMP_NTZ(6)
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED' DATA_RETENTION_TIME_IN_DAYS=1;

INSERT INTO ICEBERG_ORDER_EVENT_BATCH VALUES
  (6001, 201, '2026-09-07 10:00:00', 'SHIPPED', 1200.00, 2, 'UPSERT', '2026-09-07 12:00:00'),
  (6001, 201, '2026-09-07 10:00:00', 'DELIVERED', 1200.00, 3, 'UPSERT', '2026-09-07 13:00:00'),
  (6003, 203, '2026-09-07 14:00:00', 'PAID', 80.00, 1, 'UPSERT', '2026-09-07 14:00:00'),
  (6004, 202, '2026-09-07 14:10:00', 'CANCELLED', 35.00, 1, 'DELETE', '2026-09-07 14:10:00');

ALTER SESSION SET ERROR_ON_NONDETERMINISTIC_MERGE = TRUE;
MERGE INTO ICEBERG_ORDERS T
USING (
  SELECT * EXCLUDE RN FROM (
    SELECT B.*, ROW_NUMBER() OVER (PARTITION BY ORDER_ID ORDER BY SOURCE_VERSION DESC, EVENT_TS DESC) RN
    FROM ICEBERG_ORDER_EVENT_BATCH B
  ) WHERE RN = 1
) S
ON T.ORDER_ID = S.ORDER_ID
WHEN MATCHED AND S.OP='DELETE' AND S.SOURCE_VERSION > T.SOURCE_VERSION THEN DELETE
WHEN MATCHED AND S.OP='UPSERT' AND S.SOURCE_VERSION > T.SOURCE_VERSION THEN UPDATE SET
  CUSTOMER_ID=S.CUSTOMER_ID, ORDER_TS=S.ORDER_TS, STATUS=S.STATUS,
  ORDER_TOTAL=S.ORDER_TOTAL, SOURCE_VERSION=S.SOURCE_VERSION, UPDATED_AT=CURRENT_TIMESTAMP()
WHEN NOT MATCHED AND S.OP='UPSERT' THEN INSERT
  (ORDER_ID,CUSTOMER_ID,ORDER_TS,STATUS,ORDER_TOTAL,SOURCE_VERSION,UPDATED_AT)
VALUES (S.ORDER_ID,S.CUSTOMER_ID,S.ORDER_TS,S.STATUS,S.ORDER_TOTAL,S.SOURCE_VERSION,CURRENT_TIMESTAMP());

SELECT * FROM ICEBERG_ORDERS ORDER BY ORDER_ID;
```

Rerunning the same merge changes nothing. Without source deduplication, the default nondeterministic-merge setting rejects multiple matched source rows rather than selecting one unpredictably.

## 16. Time Travel: inspect a retained historical snapshot

Snowflake queries retained Iceberg states with `AT` or `BEFORE` using a Snowflake timestamp, offset, or statement ID. `BEFORE` excludes the referenced statement, making it precise for a known incident.

```sql
UPDATE ICEBERG_PRODUCTS
SET UNIT_PRICE = UNIT_PRICE * 100, UPDATED_AT = CURRENT_TIMESTAMP()
WHERE CATEGORY IN ('COMPUTERS', 'AUDIO');

SET ICEBERG_BAD_PRICE_ID = LAST_QUERY_ID();
SET ICEBERG_INCIDENT_TS = CURRENT_TIMESTAMP();

SELECT $ICEBERG_BAD_PRICE_ID AS BAD_STATEMENT_ID, $ICEBERG_INCIDENT_TS AS RECORDED_AT;
SELECT SKU, PRODUCT_NAME, UNIT_PRICE FROM ICEBERG_PRODUCTS ORDER BY SKU;
SELECT SKU, PRODUCT_NAME, UNIT_PRICE
FROM ICEBERG_PRODUCTS BEFORE (STATEMENT => $ICEBERG_BAD_PRICE_ID) ORDER BY SKU;

-- Other retained point forms:
-- SELECT * FROM ICEBERG_PRODUCTS AT (OFFSET => -60);
-- SELECT * FROM ICEBERG_PRODUCTS AT
--   (TIMESTAMP => '2026-09-07 10:00:00 +05:30'::TIMESTAMP_TZ);
```

An Iceberg snapshot ID exposed in format metadata is distinct from Snowflake's statement ID. Snowflake does not expose `ALTER ICEBERG TABLE ... ROLLBACK TO SNAPSHOT`. Use a historical query/clone and write the good state forward.

## 17. Clone history and roll back by writing forward

The recovery clone is independently reviewable and has its own Iceberg metadata/table UUID while initially sharing unchanged files. The merge preserves the live object, grants, policies, references, and stream relationships.

```sql
CREATE ICEBERG TABLE ICEBERG_PRODUCTS_RECOVERY
  CLONE ICEBERG_PRODUCTS BEFORE (STATEMENT => $ICEBERG_BAD_PRICE_ID);

SELECT 'LIVE' AS COPY_NAME, SUM(UNIT_PRICE) AS PRICE_TOTAL FROM ICEBERG_PRODUCTS
UNION ALL
SELECT 'RECOVERY', SUM(UNIT_PRICE) FROM ICEBERG_PRODUCTS_RECOVERY;

BEGIN TRANSACTION;
  MERGE INTO ICEBERG_PRODUCTS T
  USING ICEBERG_PRODUCTS_RECOVERY S
  ON T.SKU = S.SKU
  WHEN MATCHED THEN UPDATE SET PRODUCT_NAME=S.PRODUCT_NAME, CATEGORY=S.CATEGORY,
    UNIT_PRICE=S.UNIT_PRICE, IS_ACTIVE=S.IS_ACTIVE, UPDATED_AT=CURRENT_TIMESTAMP()
  WHEN NOT MATCHED THEN INSERT (SKU,PRODUCT_NAME,CATEGORY,UNIT_PRICE,IS_ACTIVE,UPDATED_AT)
    VALUES (S.SKU,S.PRODUCT_NAME,S.CATEGORY,S.UNIT_PRICE,S.IS_ACTIVE,CURRENT_TIMESTAMP());
COMMIT;

SELECT * FROM ICEBERG_PRODUCTS ORDER BY SKU;
```

This is a new good snapshot, not a mutation of history. With Snowflake-provided storage, a clone and source must both be permanent or both transient. Source: [Iceberg cloning considerations](https://docs.snowflake.com/en/user-guide/object-clone#cloning-and-apache-iceberg-tables).

## 18. Truncate, drop, and undrop

`TRUNCATE` is DDL and cannot be transaction-rolled back, but a retained state can be cloned from before its query ID. A dropped Iceberg table can be undropped inside retention if required storage/catalog dependencies still exist.

```sql
CREATE ICEBERG TABLE ICEBERG_PROMOTIONS_DEMO (
  PROMO_CODE STRING, DISCOUNT_PCT INT
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED' DATA_RETENTION_TIME_IN_DAYS=1;
INSERT INTO ICEBERG_PROMOTIONS_DEMO VALUES ('ICE10',10),('ICE20',20);

TRUNCATE TABLE ICEBERG_PROMOTIONS_DEMO;
SET ICEBERG_TRUNCATE_ID = LAST_QUERY_ID();
CREATE ICEBERG TABLE ICEBERG_PROMOTIONS_BEFORE_TRUNCATE
  CLONE ICEBERG_PROMOTIONS_DEMO BEFORE (STATEMENT => $ICEBERG_TRUNCATE_ID);
SELECT * FROM ICEBERG_PROMOTIONS_BEFORE_TRUNCATE ORDER BY PROMO_CODE;

DROP ICEBERG TABLE ICEBERG_PROMOTIONS_DEMO;
UNDROP ICEBERG TABLE ICEBERG_PROMOTIONS_DEMO;
```

`UNDROP ICEBERG TABLE` is not supported for catalog-linked database tables. If a current object occupies the same name, rename it first, then undrop the latest dropped version.

## 19. Schema evolution and field identity

Iceberg field IDs preserve identity through safe renames. Adding or dropping a field is metadata-oriented; existing data files are not necessarily rewritten immediately. Supported type promotion includes `int` to `long`, `float` to `double`, and decimal precision widening with unchanged scale. Each Snowflake DDL statement commits separately.

```sql
ALTER ICEBERG TABLE ICEBERG_PRODUCTS ADD COLUMN BRAND STRING;
UPDATE ICEBERG_PRODUCTS SET BRAND =
  CASE CATEGORY WHEN 'COMPUTERS' THEN 'Northstar'
                WHEN 'AUDIO' THEN 'SoundPeak'
                ELSE 'AccessoryWorks' END;

ALTER ICEBERG TABLE ICEBERG_PRODUCTS RENAME COLUMN CATEGORY TO PRODUCT_CATEGORY;
ALTER ICEBERG TABLE ICEBERG_ORDERS
  ALTER COLUMN ORDER_TOTAL SET DATA TYPE DECIMAL(18,2);

DESCRIBE ICEBERG TABLE ICEBERG_PRODUCTS;
SELECT SKU, PRODUCT_NAME, PRODUCT_CATEGORY, BRAND, UNIT_PRICE
FROM ICEBERG_PRODUCTS ORDER BY SKU;

ALTER ICEBERG TABLE ICEBERG_PRODUCTS DROP COLUMN BRAND;
```

Time Travel queries in Snowflake use the current table schema, so a retained row snapshot does not automatically restore an old column name. Coordinate DDL with views, streams, BI models, and every external engine. Source: [ALTER ICEBERG TABLE](https://docs.snowflake.com/en/sql-reference/sql/alter-iceberg-table).

## 20. Iceberg v2 and v3 practical comparison

V2 is Snowflake's compatibility-oriented default and supports positional deletes. V3 adds deletion vectors, row lineage, static initial/write defaults, `VARIANT`, geospatial types, nanosecond timestamps, and `UNKNOWN`. Format upgrades cannot be downgraded, so verify all readers and writers first. This separate table makes the v3 contract explicit.

```sql
CREATE ICEBERG TABLE ICEBERG_V3_EVENTS (
  EVENT_ID LONG,
  EVENT_STATUS STRING DEFAULT 'NEW',
  PAYLOAD VARIANT,
  FUTURE_ATTRIBUTE UNKNOWN
) ICEBERG_VERSION=3 CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED'
  DATA_RETENTION_TIME_IN_DAYS=1;

INSERT INTO ICEBERG_V3_EVENTS (EVENT_ID, PAYLOAD)
SELECT 1, PARSE_JSON('{"source":"mobile","items":[1,2]}');

SELECT EVENT_ID, EVENT_STATUS, PAYLOAD:source::STRING AS SOURCE, FUTURE_ATTRIBUTE
FROM ICEBERG_V3_EVENTS;

ALTER ICEBERG TABLE ICEBERG_V3_EVENTS
  ALTER COLUMN FUTURE_ATTRIBUTE SET DATA TYPE STRING;
ALTER ICEBERG TABLE ICEBERG_V3_EVENTS
  ALTER COLUMN EVENT_STATUS SET WRITE DEFAULT 'PENDING';

INSERT INTO ICEBERG_V3_EVENTS (EVENT_ID, PAYLOAD, FUTURE_ATTRIBUTE)
SELECT 2, PARSE_JSON('{"source":"web"}'), 'campaign-42';

SELECT * FROM ICEBERG_V3_EVENTS ORDER BY EVENT_ID;
SHOW PARAMETERS LIKE 'ICEBERG_VERSION' IN TABLE ICEBERG_V3_EVENTS;
```

Source: [Snowflake support for Iceberg v3](https://docs.snowflake.com/en/user-guide/tables-iceberg-v3-specification-support).

## 21. Hidden partitioning and partition evolution

Users filter `ORDER_TS`; Iceberg applies the stored `DAY(ORDER_TS)` transform and prunes matching files. The partition value does not need to appear in the query or table schema.

```sql
SELECT ORDER_ID, STATUS, ORDER_TOTAL
FROM ICEBERG_ORDERS
WHERE ORDER_TS >= '2026-09-07'::TIMESTAMP_NTZ
  AND ORDER_TS <  '2026-09-08'::TIMESTAMP_NTZ
ORDER BY ORDER_ID;

SHOW ICEBERG TABLES LIKE 'ICEBERG_ORDERS';
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
```

Core Iceberg allows partition evolution with multiple specs coexisting. Snowflake-managed Iceberg tables currently do not support in-place partition evolution. Build and validate a replacement table with the desired spec. Externally managed tables can expose evolution performed by an external engine after refresh.

```sql
-- Replacement pattern; run only for a planned migration.
-- CREATE ICEBERG TABLE ICEBERG_ORDERS_MONTHLY (
--   ORDER_ID LONG, CUSTOMER_ID LONG, ORDER_TS TIMESTAMP_NTZ(6), STATUS STRING,
--   ORDER_TOTAL DECIMAL(18,2), SOURCE_VERSION LONG, UPDATED_AT TIMESTAMP_LTZ(6)
-- ) PARTITION BY (MONTH(ORDER_TS), BUCKET(16, CUSTOMER_ID))
--   CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED'
-- AS SELECT * FROM ICEBERG_ORDERS;
```

Source: [Iceberg partitioning support](https://docs.snowflake.com/en/user-guide/tables-iceberg-metadata#iceberg-partitioning).

## 22. Copy-on-write, merge-on-read, delete files, and deletion vectors

| Setting | Physical behavior | Trade-off |
|---|---|---|
| `AUTO` | Snowflake chooses by version and table mode | Best starting point |
| `ENABLED` | V2 can write positional delete files; v3 can use deletion vectors when conditions permit | Lighter writes can add read/compaction work |
| `DISABLED` | Rewrites affected data files using copy-on-write | Heavier writes, simpler reads |

```sql
ALTER ICEBERG TABLE ICEBERG_PRODUCTS
  SET ICEBERG_MERGE_ON_READ_BEHAVIOR='ENABLED';
UPDATE ICEBERG_PRODUCTS SET IS_ACTIVE=FALSE WHERE SKU='ICE-SKU-HDP-01';
ALTER ICEBERG TABLE ICEBERG_PRODUCTS
  SET ICEBERG_MERGE_ON_READ_BEHAVIOR='AUTO';

ALTER ICEBERG TABLE ICEBERG_PRODUCTS SET TARGET_FILE_SIZE='16MB';
ALTER ICEBERG TABLE ICEBERG_PRODUCTS SET TARGET_FILE_SIZE='AUTO';
```

The tiny lab cannot demonstrate file-size benefit. Evaluate real ingestion frequency, engines, selective reads, object-store requests, and compaction history. The older `ENABLE_ICEBERG_MERGE_ON_READ` parameter is deprecated. Partitioned v3 tables have additional deletion-vector limitations; inspect the current support matrix before selecting a write mode.

## 23. Change tracking and streams

A Snowflake stream stores an offset into table history, not copied rows. A plain `SELECT` does not advance it. A DML consumer advances it only when its transaction commits.

```sql
CREATE STREAM ICEBERG_ORDERS_STREAM ON TABLE ICEBERG_ORDERS;
CREATE ICEBERG TABLE ICEBERG_ORDER_AUDIT (
  CAPTURED_AT TIMESTAMP_LTZ(6), ORDER_ID LONG, STATUS STRING, ORDER_TOTAL DECIMAL(18,2),
  ACTION STRING, IS_UPDATE BOOLEAN, ROW_ID STRING
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='SNOWFLAKE_MANAGED' DATA_RETENTION_TIME_IN_DAYS=1;

UPDATE ICEBERG_ORDERS SET STATUS='RETURN_REQUESTED', SOURCE_VERSION=SOURCE_VERSION+1,
  UPDATED_AT=CURRENT_TIMESTAMP() WHERE ORDER_ID=6001;
INSERT INTO ICEBERG_ORDERS VALUES
  (6005,202,CURRENT_TIMESTAMP(),'PAID',35.00,1,CURRENT_TIMESTAMP());

SELECT ORDER_ID,STATUS,METADATA$ACTION,METADATA$ISUPDATE,METADATA$ROW_ID
FROM ICEBERG_ORDERS_STREAM ORDER BY ORDER_ID,METADATA$ACTION;

BEGIN;
  INSERT INTO ICEBERG_ORDER_AUDIT
  SELECT CURRENT_TIMESTAMP(),ORDER_ID,STATUS,ORDER_TOTAL,
    METADATA$ACTION,METADATA$ISUPDATE,METADATA$ROW_ID
  FROM ICEBERG_ORDERS_STREAM;
ROLLBACK;
SELECT SYSTEM$STREAM_HAS_DATA('ICEBERG_ORDERS_STREAM') AS DATA_AFTER_ROLLBACK;

INSERT INTO ICEBERG_ORDER_AUDIT
SELECT CURRENT_TIMESTAMP(),ORDER_ID,STATUS,ORDER_TOTAL,
  METADATA$ACTION,METADATA$ISUPDATE,METADATA$ROW_ID
FROM ICEBERG_ORDERS_STREAM;

SELECT * FROM ICEBERG_ORDER_AUDIT ORDER BY CAPTURED_AT,ORDER_ID,ACTION;
```

Monitor stream staleness and schema compatibility. External-engine changes must become visible through the configured catalog path before Snowflake change consumers can reason about them.

## 24. Compaction, manifests, snapshot expiry, and the `VACUUM` comparison

For Snowflake-managed Iceberg tables, Snowflake provides data-file compaction, automatic manifest compaction, and automatic snapshot expiry. `ENABLE_DATA_COMPACTION` and `TARGET_FILE_SIZE` influence data files. Manifest compaction and snapshot expiry are automatic. `DATA_RETENTION_TIME_IN_DAYS` controls how long snapshots remain available for Snowflake Time Travel.

```sql
ALTER ICEBERG TABLE ICEBERG_ORDERS SET ENABLE_DATA_COMPACTION=TRUE;
SHOW PARAMETERS LIKE 'ENABLE_DATA_COMPACTION' IN TABLE ICEBERG_ORDERS;
SHOW PARAMETERS LIKE 'TARGET_FILE_SIZE' IN TABLE ICEBERG_ORDERS;
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN TABLE ICEBERG_ORDERS;

SELECT START_TIME,END_TIME,TABLE_NAME,CREDITS_USED,NUM_BYTES_SCANNED,NUM_ROWS_WRITTEN
FROM SNOWFLAKE.ACCOUNT_USAGE.ICEBERG_STORAGE_OPTIMIZATION_HISTORY
WHERE DATABASE_NAME = CURRENT_DATABASE()
  AND SCHEMA_NAME = CURRENT_SCHEMA()
  AND TABLE_NAME LIKE 'ICEBERG_%'
ORDER BY START_TIME DESC LIMIT 50;
```

Account Usage has latency and requires suitable imported privileges. Snowflake has no `VACUUM ICEBERG TABLE` command. Snapshot expiry reclaims uniquely referenced files asynchronously after retention. Snowflake storage handles its own files; customer external volumes also require cloud cost/protection monitoring. Externally managed tables must be compacted, expired, and cleaned by the owning external engine, followed by Snowflake refresh.

## 25. Customer storage and external-catalog templates

An administrator creates the external volume and catalog integration with cloud-specific trust. Give every managed table a unique `BASE_LOCATION`.

```sql
-- Snowflake owns Iceberg commits; files live in your cloud storage.
-- CREATE ICEBERG TABLE ICEBERG_PRODUCTS_EXTERNAL (
--   SKU STRING, PRODUCT_NAME STRING, UNIT_PRICE DECIMAL(12,2)
-- ) CATALOG='SNOWFLAKE'
--   EXTERNAL_VOLUME='YOUR_EXTERNAL_VOLUME'
--   BASE_LOCATION='iceberg/products';

-- Remote REST catalog. Write support requires an appropriate catalog integration.
-- CREATE ICEBERG TABLE ICEBERG_ORDERS_FROM_REST
--   EXTERNAL_VOLUME='YOUR_EXTERNAL_VOLUME'
--   CATALOG='YOUR_REST_CATALOG_INTEGRATION'
--   CATALOG_TABLE_NAME='iceberg_orders'
--   AUTO_REFRESH=TRUE;

-- Direct registration from metadata files is commonly read-oriented.
-- CREATE ICEBERG TABLE ICEBERG_ORDERS_FROM_FILES
--   EXTERNAL_VOLUME='YOUR_EXTERNAL_VOLUME'
--   CATALOG='YOUR_OBJECT_STORE_CATALOG_INTEGRATION'
--   METADATA_FILE_PATH='iceberg_orders/metadata/v1.metadata.json';

-- Delta Direct exposes supported Delta files through an Iceberg table interface.
-- CREATE ICEBERG TABLE ICEBERG_ORDERS_FROM_DELTA
--   CATALOG='YOUR_DELTA_CATALOG_INTEGRATION'
--   EXTERNAL_VOLUME='YOUR_EXTERNAL_VOLUME'
--   BASE_LOCATION='delta/orders/'
--   AUTO_REFRESH=TRUE;

-- Synchronize after external commits or maintenance when auto-refresh is not used.
-- ALTER ICEBERG TABLE ICEBERG_ORDERS_FROM_REST REFRESH;
-- ALTER ICEBERG TABLE ICEBERG_ORDERS_FROM_FILES
--   REFRESH 'iceberg_orders/metadata/v2.metadata.json';
```

`REFRESH` is a catalog synchronization operation, not compaction. Align it with external snapshot expiry so Snowflake never points at files the external owner has removed.

## 26. Retention and disaster protection by storage mode

| Mode | Snowflake Time Travel | Fail-safe/file protection | Physical cleanup owner |
|---|---|---|---|
| Permanent Iceberg + Snowflake storage | Configurable by edition | Snowflake Fail-safe protection | Snowflake |
| Transient Iceberg + Snowflake storage | 0 or 1 day | No Fail-safe | Snowflake |
| Snowflake catalog + customer external volume | Configured table retention | No Snowflake Fail-safe for customer files; use cloud protection | Snowflake Iceberg lifecycle plus customer cloud controls |
| External catalog | Derived from external snapshot age and Snowflake limits | External owner | External engine/catalog |

For externally managed tables, Snowflake derives retention from the smaller of `history.expire.max-snapshot-age-ms` and the applicable Snowflake edition limit. Change the external property and refresh rather than trying to set the derived value directly.

Clones share referenced files. Streams depend on retained change history. Never shorten retention or expire external snapshots without checking recovery objectives, clone dependencies, stream offsets, external-engine readers, and cloud versioning. Source: [Iceberg metadata and retention](https://docs.snowflake.com/en/user-guide/tables-iceberg-metadata).

## 27. Production decision and recovery checklist

1. Name the catalog owner and every permitted writer. Avoid independent file writers that bypass catalog commits.
2. Select v2 or v3 as an interoperability contract and test every engine.
3. Define business keys and source versions; Iceberg ACID does not deduplicate events.
4. Choose a partition transform from real filters and data volume. Record the replacement plan because managed partition evolution is limited.
5. Capture query ID, timestamp, user, and affected predicate for incidents.
6. Clone the historical state, reconcile it, and restore by a forward transaction.
7. Validate order totals against items/payments and inventory against fulfilled quantity.
8. Monitor commit conflicts, refresh lag, stream staleness, file sizes, compaction credits, and snapshot storage.
9. Coordinate schema DDL across Snowflake, Spark, Flink, Trino, views, BI, and ingestion.
10. Test `UNDROP`, Time Travel, and cloud-storage recovery before an incident.

Iceberg supplies the table-level primitives. Reliable operations still require one ownership model, deterministic writes, observability, and rehearsed recovery.

## 28. Optional cleanup

```sql
USE SCHEMA IcebergTableLifecycle;
DROP STREAM IF EXISTS ICEBERG_ORDERS_STREAM;
DROP ICEBERG TABLE IF EXISTS ICEBERG_ORDER_AUDIT;
DROP ICEBERG TABLE IF EXISTS ICEBERG_PROMOTIONS_BEFORE_TRUNCATE;
DROP ICEBERG TABLE IF EXISTS ICEBERG_PROMOTIONS_DEMO;
DROP ICEBERG TABLE IF EXISTS ICEBERG_V3_EVENTS;
DROP ICEBERG TABLE IF EXISTS ICEBERG_PRODUCTS_RECOVERY;
DROP ICEBERG TABLE IF EXISTS ICEBERG_ORDER_EVENT_BATCH;
DROP ICEBERG TABLE IF EXISTS ICEBERG_PAYMENTS;
DROP ICEBERG TABLE IF EXISTS ICEBERG_ORDER_ITEMS;
DROP ICEBERG TABLE IF EXISTS ICEBERG_ORDERS;
DROP ICEBERG TABLE IF EXISTS ICEBERG_INVENTORY;
DROP ICEBERG TABLE IF EXISTS ICEBERG_PRODUCTS;
DROP SCHEMA IF EXISTS IcebergTableLifecycle RESTRICT;
```

Run cleanup only in the original database, with the role that created the lab objects. The existing database and `SNOWFLAKE_LEARNING_WH` are retained. If optional templates created additional objects, `RESTRICT` leaves the schema in place until those objects are handled.